# Sim2Real Channel Prediction Notebook

이 노트북은 `sim2real_channel_predict.py`를 불러와서,
시뮬레이션 CIR로 학습하고 실제 CIR(`CIR.mat`)에 대해 예측/평가를 수행합니다.

In [1]:
from __future__ import annotations

from pathlib import Path
import importlib.util
from types import SimpleNamespace

script_path = Path('/home/mh/kmh/sionna-rt/jinsup/sim2real_channel_predict.py')
spec = importlib.util.spec_from_file_location('sim2real_module', script_path)
mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(mod)

print(f'loaded: {script_path}')

loaded: /home/mh/kmh/sionna-rt/jinsup/sim2real_channel_predict.py


In [2]:
# ===== 실험 설정 =====
sim_path = '/home/mh/kmh/sionna-rt/jinsup/cir_pdp_exports'
real_path = '/data/kmh/sionna-rt/mh/workspace/data_analyze/data/CIR.mat'

args = SimpleNamespace(
    sim_path=sim_path,
    real_path=real_path,
    sim_key='h_cir_noisy',
    real_key='CIR',
    sim_layout='auto',
    real_layout='auto',
    window=8,
    horizon=1,
    max_delay=256,
    alpha=1.0,
    calib_ratio=0.25,
    out_npz='/data/kmh/sionna-rt/jinsup/cir_pdp_exports/sim2real_eval_from_notebook.npz',
)

args

namespace(sim_path='/home/mh/kmh/sionna-rt/jinsup/cir_pdp_exports',
          real_path='/data/kmh/sionna-rt/mh/workspace/data_analyze/data/CIR.mat',
          sim_key='h_cir_noisy',
          real_key='CIR',
          sim_layout='auto',
          real_layout='auto',
          window=8,
          horizon=1,
          max_delay=256,
          alpha=1.0,
          calib_ratio=0.25,
          out_npz='/data/kmh/sionna-rt/jinsup/cir_pdp_exports/sim2real_eval_from_notebook.npz')

In [3]:
# 실행
mod.run(args)

[sim] loaded shape=(281, 4095), key=h_cir_noisy
[real] loaded shape=(281, 4095), key=CIR
[common] using delay taps=256
[base] trained with samples=273, in_dim=4096, out_dim=512
[split] real windows=273, calibration=68, eval=205, window=8, horizon=1
Base(sim-only): MSE=3.822977e-05, MAE=5.030173e-03, NMSE=2.057235e+00
Sim2Real(calibrated): MSE=3.893722e-05, MAE=5.095592e-03, NMSE=2.095305e+00
Persistence: MSE=3.710577e-05, MAE=5.370588e-03, NMSE=1.996750e+00
[save] /data/kmh/sionna-rt/jinsup/cir_pdp_exports/sim2real_eval_from_notebook.npz


## 결과 해석 가이드
- `MSE`, `MAE`, `NMSE`는 **작을수록 좋음**
- `Persistence`(직전 프레임 복사)보다 `Sim2Real(calibrated)`가 낮아야 개선
- `horizon=1`은 Persistence가 강하므로, `horizon=4` 또는 `8`도 같이 테스트 권장